# Imports

In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from rdkit import Chem
import json
import time
import requests
import re

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths_and_constants import Paths, Constants
_P = Paths()
_C = Constants()

# Functions

In [5]:
import json
import time
from pathlib import Path
import requests

def sequence_getter(
    uniprot_ids,
    checkpoint_path="uniprot_checkpoint.json",
    save_every=50,
    timeout=15,
    sleep_s=0.0,
    retry_failed=True,
    show_progress=True,
):
    """
    Download UniProt sequences with checkpoint resume.
    Fetch order:
    1) direct FASTA by accession
    2) direct JSON by accession
    3) UniProt search fallback (accession/id/secondary accession)
    Returns:
      done   -> {input_id: sequence}
      failed -> {input_id: error_message}
    """
    ckpt = Path(checkpoint_path)

    if ckpt.exists():
        with ckpt.open("r", encoding="utf-8") as f:
            state = json.load(f)
        done = state.get("done", {})
        failed = state.get("failed", {})
    else:
        done, failed = {}, {}

    ids = []
    seen = set()
    for x in uniprot_ids:
        uid = str(x).strip()
        if uid and uid.lower() != "nan" and uid not in seen:
            ids.append(uid)
            seen.add(uid)

    if retry_failed:
        for uid in ids:
            failed.pop(uid, None)

    pending = [uid for uid in ids if uid not in done and uid not in failed]

    total = len(ids)
    pending_total = len(pending)
    overall_done = total - pending_total

    def save_ckpt():
        tmp = ckpt.with_suffix(ckpt.suffix + ".tmp")
        with tmp.open("w", encoding="utf-8") as f:
            json.dump({"done": done, "failed": failed}, f, ensure_ascii=False, indent=2)
        tmp.replace(ckpt)

    if show_progress:
        print(
            f"Resume status -> done: {len(done)}, failed: {len(failed)}, "
            f"pending: {pending_total}, total: {total}"
        )

    with requests.Session() as session:
        for i, uid in enumerate(pending, start=1):
            try:
                seq = ""

                # 1) Direct FASTA
                r = session.get(f"https://rest.uniprot.org/uniprotkb/{uid}.fasta", timeout=timeout)
                if r.status_code == 200:
                    lines = r.text.strip().splitlines()
                    if lines and lines[0].startswith(">"):
                        seq = "".join(lines[1:]).replace(" ", "").upper()

                # 2) Direct JSON
                if not seq:
                    rj = session.get(f"https://rest.uniprot.org/uniprotkb/{uid}.json", timeout=timeout)
                    if rj.status_code == 200:
                        seq = rj.json().get("sequence", {}).get("value", "").upper()

                # 3) Search fallback (covers secondary accessions / renamed entries)
                if not seq:
                    rs = session.get(
                        "https://rest.uniprot.org/uniprotkb/search",
                        params={
                            "query": f"(accession:{uid}) OR (id:{uid}) OR (sec_acc:{uid})",
                            "format": "json",
                            "size": 5,
                        },
                        timeout=timeout,
                    )
                    rs.raise_for_status()
                    results = rs.json().get("results", [])

                    chosen = None
                    for entry in results:
                        acc = entry.get("primaryAccession", "")
                        kb_id = entry.get("uniProtkbId", "")
                        sec = set(entry.get("secondaryAccessions", []))
                        if uid == acc or uid == kb_id or uid in sec:
                            chosen = entry
                            break
                    if chosen is None and results:
                        chosen = results[0]

                    if chosen:
                        seq = chosen.get("sequence", {}).get("value", "").upper()

                if not seq:
                    raise ValueError(f"No sequence found for: {uid}")

                done[uid] = seq
                failed.pop(uid, None)

            except Exception as e:
                failed[uid] = str(e)

            if i % save_every == 0:
                save_ckpt()

            if show_progress and (i % save_every == 0 or i == pending_total):
                print(
                    f"Progress pending: {i}/{pending_total} ({(i/pending_total*100 if pending_total else 100):.2f}%) | "
                    f"overall: {overall_done + i}/{total} ({((overall_done + i)/total*100 if total else 100):.2f}%)"
                )

            if sleep_s > 0:
                time.sleep(sleep_s)

    save_ckpt()
    return done, failed

MUT_RE = re.compile(r"^([A-Z])(\d+)([A-Z])$")

def parse_substitution(sub):
    """
    Convert 'A123G' -> ('A', 123, 'G')
    """
    m = MUT_RE.match(sub.strip().upper())
    if not m:
        raise ValueError(f"Substitution format invalid: {sub}")
    wt, pos, mut = m.groups()
    return wt, int(pos), mut

def apply_substitutions(seq, substitutions, sep=r"[;,]\s*"):
    """
    Applies one or multiple substitutions to a sequence.
    - seq: original sequence (string)
    - substitutions: 'A123G' o 'A123G;L130P'
    - sep: separator between multiple substitutions
    """
    seq_list = list(seq)
    muts = re.split(sep, substitutions.strip())

    for sub in muts:
        if not sub:
            continue
        wt, pos, mut = parse_substitution(sub)
        idx = pos - 1  # 1-based -> 0-based

        if idx < 0 or idx >= len(seq_list):
            raise IndexError(f"Position out of range: {pos} for sequence of length {len(seq_list)}")

        if seq_list[idx] != wt:
            raise ValueError(
                f"Mismatch in {sub}: expected '{wt}' at position {pos}, "
                f"but found '{seq_list[idx]}'"
            )

        seq_list[idx] = mut

    return "".join(seq_list)

def peek_mutations(wt_seq, mut_seq, ignore_case=True):
    """
    Compares wt_seq vs mut_seq y devuelve una lista de mutaciones tipo A23L.
    """
    if ignore_case:
        wt_seq = wt_seq.upper()
        mut_seq = mut_seq.upper()

    if len(wt_seq) != len(mut_seq):
        raise ValueError(
            f"Sequences have different lengths: {len(wt_seq)} vs {len(mut_seq)}"
        )

    muts = []
    for i, (wt_aa, mut_aa) in enumerate(zip(wt_seq, mut_seq), start=1):  # 1-based
        if wt_aa != mut_aa:
            muts.append(f"{wt_aa}{i}{mut_aa}")
    return muts



# Propertyprep for binding (biolip)

In [40]:
biol = pd.read_csv(_P.data_processed_datasets / 'biolip_binding' / 'biolip_cleaned_with_negatives.csv', index_col=0)
print(biol.columns)

biol["protein_sequence"] = biol["protein_sequence"].astype(str).str.strip()
biol["uniprot_id"] = biol["uniprot_id"].replace("", pd.NA)

seq_to_uniprot = (
    biol.dropna(subset=["uniprot_id"])
      .drop_duplicates(subset=["protein_sequence"])
      .set_index("protein_sequence")["uniprot_id"]
)

biol["uniprot_id"] = biol["uniprot_id"].fillna(biol["protein_sequence"].map(seq_to_uniprot))

Index(['chain_id', 'resolution', 'binding_site_id', 'ligand_id',
       'ligand_chain_id', 'ligand_serial_number', 'binding_site_residues',
       'nearby_residues', 'ligand_contact_residues', 'catalytic_site_residues',
       'ec_number', 'go_terms', 'binding_affinity_1', 'binding_affinity_2',
       'binding_affinity_3', 'binding_affinity_4', 'uniprot_id', 'pubmed_id',
       'binding_site_range', 'protein_sequence', 'ligand_id_raw', 'smiles',
       'ligand_name', 'map_status', 'label'],
      dtype='object')


/tmp/ipykernel_18933/1070420007.py:1: DtypeWarning: Columns (13,14,16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  biol = pd.read_csv(_P.data_processed_datasets / 'biolip_binding' / 'biolip_cleaned_with_negatives.csv', index_col=0)


In [32]:
biol['smiles'] = biol['smiles'].astype(str).str.strip()
biol['ligand_id_raw'] = biol['ligand_id_raw'].replace("", pd.NA)
biol['ligand_name'] = biol['ligand_name'].replace("", pd.NA)

smiles_to_id = (
    biol.dropna(subset=['ligand_id_raw'])
      .drop_duplicates(subset=['smiles'])
      .set_index('smiles')['ligand_id_raw']
)

smiles_to_name = (
    biol.dropna(subset=['ligand_name'])
      .drop_duplicates(subset=['smiles'])
      .set_index('smiles')['ligand_name']
)

biol['ligand_id_raw'] = biol['ligand_id_raw'].fillna(biol['smiles'].map(smiles_to_id))
biol['ligand_name'] = biol['ligand_name'].fillna(biol['smiles'].map(smiles_to_name))

In [42]:
final_cols = ['uniprot_id', 'protein_sequence', 'ligand_id', 'ligand_name', 'smiles', 'label']
biol_final = biol[final_cols]
biol_final = biol_final.rename(columns={'label': 'binding'})
biol_final = biol_final.reset_index(drop=True)
biol_final.to_csv(_P.data_processed_properties / 'binding' / 'biolip_binding.csv', index=False)

# Propertyprep for DDG (FireProtDB)

In [11]:
ddg = pd.read_csv(_P.data_interim_dir / 'fireprotdb' / 'fireprotdb_ddg.csv', index_col=0)
ddg = ddg.reset_index()
print(ddg.columns)

Index(['EXPERIMENT_ID', 'SEQUENCE_ID', 'MUTANT_ID', 'SOURCE_SEQUENCE_ID',
       'TARGET_SEQUENCE_ID', 'SEQUENCE_LENGTH', 'SUBSTITUTION', 'INSERTION',
       'DELETION', 'PROTEIN', 'ORGANISM', 'CHYMOTRYPSIN_ML', 'CM', 'DCP',
       'DDG', 'DG', 'DH', 'DHVH', 'DOMAINOME_DDG', 'DOMAINOME_DDG_STD',
       'DOMAINOME_FITNESS', 'DOMAINOME_FITNESS_STD', 'DTM', 'M',
       'REVERSIBILITY', 'STABILIZING', 'STATE', 'TM', 'TRYPSIN_ML', 'BUFFER',
       'BUFFER_CONC', 'EXP_TEMPERATURE', 'ION', 'ION_CONC', 'MEASURE',
       'METHOD', 'PH', 'EC_NUMBER', 'UNIPROTKB', 'INTERPRO', 'MEGASCALE',
       'ACTIVE_SITE', 'BINDING_SITE', 'CONSERVATION', 'WWPDB', 'B_FACTOR',
       'IN_TUNNEL', 'IN_POCKET', 'PUBLICATION_PMID', 'PUBLICATION_DOI',
       'PUBLICATION_YEAR', 'SOURCE_DATASET', 'REFERENCING_DATASET'],
      dtype='object')


In [12]:
fire_cols = ['EXPERIMENT_ID', 'UNIPROTKB', 'SUBSTITUTION', 'DDG']
ddg = ddg[fire_cols]
unique_prot = ddg['UNIPROTKB'].unique()
unique_prot

array(['P0A877', 'P00651', 'P69441', 'P00720', 'P61626', 'P00698',
       'P00929', 'P00644', 'P33665', 'P0A7Y4', 'P29957', 'P00648',
       'P62593', 'P29768', 'P81708', 'P06241', 'P61823', 'P05798',
       'P03706', 'Q9REI6', 'P96110', 'P04179', 'P11961', 'P22069',
       'O74035', 'P00918', 'P0AFG6', 'Q13526', 'P00782', 'P00563',
       'P0ABQ4', 'P00750', 'P00811', 'P07170', 'Q16595', 'P0C2S3',
       'P23540', 'O61594', 'P11309', 'P16304', 'O49003', 'Q8TZE8',
       'P00593', 'P07311', 'P30289', 'P07445', 'P62942', 'Q8WZ42',
       'P00509', 'P0A434', 'P37001', 'P00560', 'P18912', 'P00952',
       'O95363', 'P00558', 'P20533', 'P06213', 'P00912', 'P00807',
       'P03958', 'P00592', 'Q23551', 'P15917', 'P35670', 'P38398',
       'P41500', 'P09850', 'Q88IY3'], dtype=object)

In [13]:
done, failed = sequence_getter(
    unique_prot,
    checkpoint_path=str(_P.data_interim_dir / "sequence_checkpoint" / "uniprot_checkpoint.json"),
    save_every=10
)

Resume status -> done: 11282, failed: 355, pending: 0, total: 69


In [15]:
ddg["sequence"] = ddg['UNIPROTKB'].map(done)
ddg_f = ddg.rename(columns={
    'EXPERIMENT_ID': _C.dataset_id,
    'UNIPROTKB': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'SUBSTITUTION': 'mutation',
    'DDG': 'ddg'
})
ddg_f['mut_protein_sequence'] = ddg_f['protein_sequence'] #creating new column to then change to a mutated sequence
ddg_f['mut_protein_sequence'] = ddg_f.apply(
    lambda row: apply_substitutions(row['protein_sequence'], row['mutation']), axis=1) #changing to mutated sequence
print(ddg_f.columns)
ddg_f.rename(columns={'protein_sequence': 'wildtype_sequence', 'mut_protein_sequence': 'protein_sequence'}, inplace=True)
print(ddg_f.columns)
ddg_f = ddg_f[
    ["dataset_id", "uniprot_id", "wildtype_sequence", "mutation", "protein_sequence", "ddg"]
]
print(ddg_f.columns)
#Sanity check: peek mutations from wildtype vs mutated sequence and compare with reported mutation
a = ddg_f.loc[0, "wildtype_sequence"]
b = ddg_f.loc[0, "protein_sequence"]
print(a)
print(b)
print(peek_mutations(a, b))

print(ddg_f.shape)
ddg_f.dropna(subset=[_C.protein_sequence, 'ddg'], inplace=True)
print(ddg_f.shape)
ddg_f = ddg_f.drop_duplicates(subset=[_C.protein_sequence, 'ddg'])
print(ddg_f.shape)
ddg_f.to_csv(_P.data_processed_properties / 'ddg_mut' / 'fireprotdb_ddg.csv', index=False)

Index(['dataset_id', 'uniprot_id', 'mutation', 'ddg', 'protein_sequence',
       'mut_protein_sequence'],
      dtype='object')
Index(['dataset_id', 'uniprot_id', 'mutation', 'ddg', 'wildtype_sequence',
       'protein_sequence'],
      dtype='object')
Index(['dataset_id', 'uniprot_id', 'wildtype_sequence', 'mutation',
       'protein_sequence', 'ddg'],
      dtype='object')
MERYESLFAQLKERKEGAFVPFVTLGDPGIEQSLKIIDTLIEAGADALELGIPFSDPLADGPTIQNATLRAFAAGVTPAQCFEMLALIRQKHPTIPIGLLMYANLVFNKGIDEFYAQCEKVGVDSVLVADVPVEESAPFRQAALRHNVAPIFICPPNADDDLLRQIASYGRGYTYLLSRAGVTGAENRAALPLNHLVAKLKEYNAAPPLQGFGISAPDQVKAAIDAGAAGAISGSAIVKIIEQHINEPEKMLAALKVFVQPMKAATRS
MERYESLFAQLKERKEGAFVPLVTLGDPGIEQSLKIIDTLIEAGADALELGIPFSDPLADGPTIQNATLRAFAAGVTPAQCFEMLALIRQKHPTIPIGLLMYANLVFNKGIDEFYAQCEKVGVDSVLVADVPVEESAPFRQAALRHNVAPIFICPPNADDDLLRQIASYGRGYTYLLSRAGVTGAENRAALPLNHLVAKLKEYNAAPPLQGFGISAPDQVKAAIDAGAAGAISGSAIVKIIEQHINEPEKMLAALKVFVQPMKAATRS
['F22L']
(3992, 6)
(3992, 6)
(3774, 6)


# Propertyprep for dtm (FireProtDB)

In [16]:
dtm = pd.read_csv(_P.data_interim_dir / 'fireprotdb' / 'fireprotdb_dtm.csv', index_col=0)
dtm = dtm.reset_index()
print(dtm.columns)
fire_cols = ['EXPERIMENT_ID', 'UNIPROTKB', 'SUBSTITUTION', 'DTM']
dtm = dtm[fire_cols]
unique_prot_dtm = dtm['UNIPROTKB'].unique()
unique_prot_dtm

Index(['EXPERIMENT_ID', 'SEQUENCE_ID', 'MUTANT_ID', 'SOURCE_SEQUENCE_ID',
       'TARGET_SEQUENCE_ID', 'SEQUENCE_LENGTH', 'SUBSTITUTION', 'INSERTION',
       'DELETION', 'PROTEIN', 'ORGANISM', 'CHYMOTRYPSIN_ML', 'CM', 'DCP',
       'DDG', 'DG', 'DH', 'DHVH', 'DOMAINOME_DDG', 'DOMAINOME_DDG_STD',
       'DOMAINOME_FITNESS', 'DOMAINOME_FITNESS_STD', 'DTM', 'M',
       'REVERSIBILITY', 'STABILIZING', 'STATE', 'TM', 'TRYPSIN_ML', 'BUFFER',
       'BUFFER_CONC', 'EXP_TEMPERATURE', 'ION', 'ION_CONC', 'MEASURE',
       'METHOD', 'PH', 'EC_NUMBER', 'UNIPROTKB', 'INTERPRO', 'MEGASCALE',
       'ACTIVE_SITE', 'BINDING_SITE', 'CONSERVATION', 'WWPDB', 'B_FACTOR',
       'IN_TUNNEL', 'IN_POCKET', 'PUBLICATION_PMID', 'PUBLICATION_DOI',
       'PUBLICATION_YEAR', 'SOURCE_DATASET', 'REFERENCING_DATASET'],
      dtype='object')


array(['P0A877', 'P00651', 'P69441', 'P00720', 'P61626', 'P00698',
       'P00929', 'P00644', 'P61495', 'P33665', 'P00883', 'P0A7Y4',
       'P29957', 'P00648', 'P62593', 'P29768', 'P81708', 'P06241',
       'P26514', 'Q43763', 'P61823', 'P05798', 'P03706', 'P04789',
       'P41365', 'P05645', 'Q9REI6', 'P80040', 'P00942', 'P47189',
       'P96110', 'P04179', 'P11961', 'P24058', 'P22069', 'O68541',
       'P00183', 'O74035', 'P00918', 'P50455', 'Q13526', 'P00782',
       'P00563', 'P21549', 'Q9L9D7', 'P06547', 'P00445', 'P06182',
       'P0ABQ4', 'P00889', 'Q56304', 'P00750', 'P00749', 'P00441',
       'P00943', 'Q56319', 'P07170', 'P25984', 'P08038', 'P28366',
       'P10408', 'P60484', 'Q16595', 'Q5S007', 'Q5KZ26', 'P48842',
       'P14756', 'P0A115', 'D4Z2G1', 'Q9ZEU2', 'O25949', 'P23458',
       'P60953', 'P0C2S3', 'P23540', 'P18429', 'P19515', 'O61594',
       'Q8NBP7', 'P26882', 'P16304', 'O95571', 'Q9ZAG3', 'O49003',
       'P11413', 'A0A060GYS1', 'P63096', 'P39940', 'Q54276', '

In [17]:
done, failed = sequence_getter(
    unique_prot_dtm,
    checkpoint_path=str(_P.data_interim_dir / "sequence_checkpoint" / "uniprot_checkpoint.json"),
    save_every=10
)

Resume status -> done: 11282, failed: 355, pending: 0, total: 92


In [20]:
dtm["sequence"] = dtm['UNIPROTKB'].map(done)
dtm_f = dtm.rename(columns={
    'EXPERIMENT_ID': _C.dataset_id,
    'UNIPROTKB': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'SUBSTITUTION': 'mutation',
    'DTM': 'dtm'
})
print(dtm_f.shape)
dtm_f = dtm_f.dropna(subset=['mutation'])
print(dtm_f.shape)

(3514, 5)
(3509, 5)


In [21]:
dtm_f['mut_protein_sequence'] = dtm_f['protein_sequence']
dtm_f['mut_protein_sequence'] = dtm_f.apply(lambda row: apply_substitutions(row['protein_sequence'], row['mutation']), axis=1)
print(dtm_f.columns)
dtm_f.rename(columns={'protein_sequence': 'wildtype_sequence', 'mut_protein_sequence': 'protein_sequence'}, inplace=True)
print(dtm_f.columns)
dtm_f = dtm_f[
    ["dataset_id", "uniprot_id", "wildtype_sequence", "mutation", "protein_sequence", "dtm"]
]
print(dtm_f.columns)
print(dtm_f.shape)
dtm_f = dtm_f.drop_duplicates(subset=[_C.protein_sequence, 'dtm'])
print(dtm_f.shape)
dtm_f.to_csv(_P.data_processed_properties / 'dtm' / 'fireprotdb_dtm.csv', index=False)

Index(['dataset_id', 'uniprot_id', 'mutation', 'dtm', 'protein_sequence',
       'mut_protein_sequence'],
      dtype='object')
Index(['dataset_id', 'uniprot_id', 'mutation', 'dtm', 'wildtype_sequence',
       'protein_sequence'],
      dtype='object')
Index(['dataset_id', 'uniprot_id', 'wildtype_sequence', 'mutation',
       'protein_sequence', 'dtm'],
      dtype='object')
(3509, 6)
(3338, 6)


# Polypropertypreb phopt (BRENDA)

In [2]:
brenda_raw = pd.read_csv(_P.data_interim_dir / 'phopt' / 'brenda_phopt.csv', index_col=0)
print(brenda_raw.columns)
print(brenda_raw.shape)
print(len(brenda_raw['uniprot_from_brenda'].unique()))
unique_brenda = brenda_raw['uniprot_from_brenda'].unique()

Index(['protein_id', 'organism', 'uniprot_from_brenda', 'phopt_median',
       'n_measurements', 'phopt_std'],
      dtype='object')
(12369, 6)
11557


In [6]:
done, failed = sequence_getter(
    unique_brenda,
    checkpoint_path=str(_P.data_interim_dir / "sequence_checkpoint" / "uniprot_checkpoint.json"),
    save_every=10
)

Resume status -> done: 11282, failed: 0, pending: 355, total: 11557
Progress pending: 10/355 (2.82%) | overall: 11212/11557 (97.01%)
Progress pending: 20/355 (5.63%) | overall: 11222/11557 (97.10%)
Progress pending: 30/355 (8.45%) | overall: 11232/11557 (97.19%)
Progress pending: 40/355 (11.27%) | overall: 11242/11557 (97.27%)
Progress pending: 50/355 (14.08%) | overall: 11252/11557 (97.36%)
Progress pending: 60/355 (16.90%) | overall: 11262/11557 (97.45%)
Progress pending: 70/355 (19.72%) | overall: 11272/11557 (97.53%)
Progress pending: 80/355 (22.54%) | overall: 11282/11557 (97.62%)
Progress pending: 90/355 (25.35%) | overall: 11292/11557 (97.71%)
Progress pending: 100/355 (28.17%) | overall: 11302/11557 (97.79%)
Progress pending: 110/355 (30.99%) | overall: 11312/11557 (97.88%)
Progress pending: 120/355 (33.80%) | overall: 11322/11557 (97.97%)
Progress pending: 130/355 (36.62%) | overall: 11332/11557 (98.05%)
Progress pending: 140/355 (39.44%) | overall: 11342/11557 (98.14%)
Progre

In [10]:
cols = ['uniprot_from_brenda', 'phopt_median','n_measurements', 'phopt_std']
brenda_opt = brenda_raw[cols]
brenda_opt["sequence"] = brenda_opt['uniprot_from_brenda'].map(done)
brenda_opt = brenda_opt.rename(columns={
    'uniprot_from_brenda': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'phopt': 'phopt'
})
print(brenda_opt.shape)
brenda_opt = brenda_opt[[_C.uniprot_id, _C.protein_sequence, 'phopt_median', 'n_measurements', 'phopt_std']]
brenda_opt =  brenda_opt.dropna(subset=[_C.protein_sequence])
print(brenda_opt.shape)
brenda_opt = brenda_opt.drop_duplicates(subset=[_C.protein_sequence, 'phopt_median', 'n_measurements', 'phopt_std'])
print(brenda_opt.shape)
brenda_opt.to_csv(_P.data_processed_properties / 'ph_opt' / 'brenda_phopt.csv', index=False)

(12369, 5)
(11989, 5)
(11464, 5)


/tmp/ipykernel_13060/156576799.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  brenda_opt["sequence"] = brenda_opt['uniprot_from_brenda'].map(done)


# Propertyprep for kcat (CatPred)

In [89]:
kcat_raw = pd.read_csv(_P.data_raw_catpred_dir / 'CatPred_kcat.csv', index_col=0)
kcat_columns = ['uniprot', 'sequence', 'reactant_smiles', 'temperature', 'ph', 'log10kcat_max']
kcat = kcat_raw[kcat_columns]
kcat = kcat.rename(columns={
    'uniprot': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'reactant_smiles': _C.smiles,
    'temperature': 'temperature',
    'ph': 'ph',
    'log10kcat_max': 'kcat'
})
kcat.to_csv(_P.data_processed_properties / 'kcat' / 'catpred_kcat.csv', index=False)

# Propertyprep for km (CatPred)

In [12]:
km_raw = pd.read_csv(_P.data_raw_catpred_dir / 'CatPred_km.csv', index_col=0)
km_columns = ['uniprot', 'sequence', 'substrate_smiles', 'temperature', 'ph', 'log10km_mean']
km = km_raw[km_columns]
km = km.rename(columns={
    'uniprot': _C.uniprot_id,
    'sequence': _C.protein_sequence,
    'substrate_smiles': _C.smiles,
    'temperature': 'temperature',
    'ph': 'ph',
    'log10km_mean': 'km'
})
km.to_csv(_P.data_processed_properties / 'km' / 'catpred_km.csv', index=False)

# Propertyprep for EC number (UniProt)

In [23]:
ec = pd.read_csv(_P.data_raw_swissprot_dir / 'swissprot_ec.tsv', index_col=1, sep='\t')
ec_columns = ['Entry', 'Sequence', 'EC number']
ec = ec[ec_columns]
ec = ec.rename(columns={
    'Entry': _C.uniprot_id,
    'Sequence': _C.protein_sequence,
    'EC number': 'ec_number'
})
print(ec.shape)
ec = ec.dropna(subset=['ec_number'])
print(ec.shape)
ec = ec.drop_duplicates(subset=[_C.protein_sequence, 'ec_number'])
print(ec.shape)
ec.to_csv(_P.data_processed_properties / 'ec_number' / 'swissprot_ec.csv', index=False)

(279501, 3)
(279501, 3)
(237526, 3)


# Propertyprep for active sites (UniProt)

In [94]:
active = pd.read_csv(_P.data_raw_swissprot_dir / 'swissprot_ec.tsv', index_col=1, sep='\t')
print(active.columns)
active_columns = ['Entry', 'EC number', 'Sequence', 'Active site']
active = active[active_columns]
active = active.rename(columns={
    'Entry': _C.uniprot_id,
    'EC number': 'ec_number',
    'Sequence': _C.protein_sequence,
    'Active site': 'active_site'
})
print(active.shape)
active.dropna(subset=['active_site'], inplace=True)
print(active.shape)
active.drop_duplicates(subset=[_C.protein_sequence, 'active_site'], inplace=True)
print(active.shape)

Index(['Entry', 'Protein names', 'EC number', 'Gene Ontology IDs',
       'Gene Ontology (GO)', 'Organism (ID)', 'Organism', 'Protein existence',
       'Gene Names', 'Sequence', 'Length', 'Function [CC]',
       'Catalytic activity', 'Pathway', 'Cofactor', 'Activity regulation',
       'Active site', 'Binding site', 'PDB', 'KEGG'],
      dtype='object')
(279501, 4)
(104779, 4)
(90436, 4)


In [95]:
active.to_csv(_P.data_interim_dir / 'swissprot' / 'swissprot_active_site.csv', index=False)

In [98]:
pattern = re.compile(r"ACT_SITE\s+(\d+)")

def parse_active_sites(v):
    # NaN / None
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []

    # if its parsead as list/tuple/set
    if isinstance(v, (list, tuple, set)):
        out = []
        for x in v:
            try:
                out.append(int(x))
            except (TypeError, ValueError):
                pass
        return sorted(set(out))

    # if its a single number
    if isinstance(v, (int, float)) and not pd.isna(v):
        return [int(v)]

    #normal: string
    s = str(v)
    return [int(x) for x in pattern.findall(s)]

active['active_site'] = active['active_site'].apply(parse_active_sites)
active.to_csv(_P.data_processed_properties / 'active_sites' / 'swissprot_active_site.csv', index=False)

# Propertyprep for binding sites (UniProt)

In [107]:
binding_site = pd.read_csv(_P.data_raw_swissprot_dir / 'swissprot_ec.tsv', index_col=1, sep='\t')
print(binding_site.columns)
binding_columns = ['Entry', 'EC number', 'Sequence', 'Binding site']
binding_site = binding_site[binding_columns]
binding_site = binding_site.rename(columns={
    'Entry': _C.uniprot_id,
    'EC number': 'ec_number',
    'Sequence': _C.protein_sequence,
    'Binding site': 'binding_site'
})
print(binding_site.shape)
binding_site.dropna(subset=['binding_site'], inplace=True)
print(binding_site.shape)
binding_site.drop_duplicates(subset=[_C.protein_sequence, 'binding_site'], inplace=True)
print(binding_site.shape)

Index(['Entry', 'Protein names', 'EC number', 'Gene Ontology IDs',
       'Gene Ontology (GO)', 'Organism (ID)', 'Organism', 'Protein existence',
       'Gene Names', 'Sequence', 'Length', 'Function [CC]',
       'Catalytic activity', 'Pathway', 'Cofactor', 'Activity regulation',
       'Active site', 'Binding site', 'PDB', 'KEGG'],
      dtype='object')
(279501, 4)
(185292, 4)
(158681, 4)


In [108]:
binding_site.to_csv(_P.data_interim_dir / 'swissprot' / 'swissprot_binding_site.csv', index=False)

In [109]:
seg_re = re.compile(r'BINDING\s+([^;]+);\s*(.*?)(?=BINDING\s+|$)', re.DOTALL)
lig_re = re.compile(r'/ligand="([^"]*)"')
chebi_re = re.compile(r'/ligand_id="([^"]*)"')
int_re = re.compile(r'^\d+$')
range_re = re.compile(r'^(\d+)\.\.(\d+)$')

def parse_loc_safe(loc: str):
    """
    Returns (isoform, positions_list) or (isoform, None) if it cannot be parsed.
    Handles:
      - "414"
      - "1696..1703"
      - "P12821-3:414"
      - "P12821-3:1696..1703"
    Ignores:
      - "<1", ">300", "?", "?..?", etc.
    """
    loc = loc.strip()

    isoform = None
    if ":" in loc:
        isoform, loc = loc.split(":", 1)
        loc = loc.strip()

    # numeric range case
    m = range_re.match(loc)
    if m:
        start, end = int(m.group(1)), int(m.group(2))
        if start <= end:
            return isoform, list(range(start, end + 1))
        return isoform, None

    # numeric single value case
    if int_re.match(loc):
        return isoform, [int(loc)]

    # everything else: not parseable (<1, >N, ?, etc.)
    return isoform, None


def parse_binding(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return [], [], [], 0  # + ignored_count

    s = str(v)

    positions = []
    ligands = []
    ligand_ids = []
    ignored = 0

    for m in seg_re.finditer(s):
        loc_raw = m.group(1).strip()
        attrs = m.group(2)

        _, pos_list = parse_loc_safe(loc_raw)
        if not pos_list:
            ignored += 1
            continue

        ligand = lig_re.search(attrs)
        chebi = chebi_re.search(attrs)

        ligand_val = ligand.group(1) if ligand else None
        chebi_val = chebi.group(1) if chebi else None

        for p in pos_list:
            positions.append(p)
            ligands.append(ligand_val)
            ligand_ids.append(chebi_val)

    return positions, ligands, ligand_ids, ignored


parsed = binding_site["binding_site"].apply(parse_binding)
binding_site["binding_positions"] = parsed.apply(lambda x: x[0])
binding_site["binding_ligand"] = parsed.apply(lambda x: x[1])
binding_site["binding_ligand_id"] = parsed.apply(lambda x: x[2])
binding_site["binding_ignored_count"] = parsed.apply(lambda x: x[3])
binding_site.to_csv(_P.data_interim_dir / 'swissprot' / 'swissprot_binding_site2.csv', index=False)

In [110]:
binding_site.drop(columns=["binding_ignored_count", "binding_site", "binding_ligand", "binding_ligand_id", "ec_number"], inplace=True)
binding_site.to_csv(_P.data_processed_properties / 'binding_site' / 'swissprot_binding_site.csv', index=False)